# 06 Option Pricing and Expiry Selection

Convert the Module 04 snapshot into synthetic next-close option terms. The pricing and integer-sizing functions are shared with Module 07. This is an illustration at the saved preview date, not a second backtest.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.project_io import OUTPUT_DIR, initialize, load_config, load_frame, save_frame, save_json

cfg = load_config()


## 2. Load snapshot and pricing inputs

Expiry is signal session plus H observed sessions. The remaining maturity at next-close entry is H minus one sessions.


In [ ]:
from src.execution import entry_option_terms, budgeted_size
from scripts.run_research import read_series
snapshot = load_frame('signal_snapshot')
test_prices = load_frame('test_prices')
volatility = load_frame('oos_ewma_volatility')
rates = read_series(OUTPUT_DIR / 'inputs/rates.parquet')
display(snapshot)


## 3. Price and size each preview

The illustrative budget is 5% of initial capital by default. Module 07 instead uses the actual available cash and current marked equity, and enforces the global position cap.


In [ ]:
rows = []
for row in snapshot.itertuples():
    i = test_prices.index.get_loc(row.signal_date)
    if row.horizon <= 1 or i + row.horizon >= len(test_prices):
        rows.append(dict(pair=row.pair, status='no_tradable_remaining_maturity'))
        continue
    entry_date = test_prices.index[i + 1]
    expiry_date = test_prices.index[i + int(row.horizon)]
    instruction = dict(pair=row.pair, dependent=row.dependent, independent=row.independent,
        signal_date=row.signal_date, expiry_date=expiry_date, direction=row.direction)
    spots, vols, rf, types, prices, deltas = entry_option_terms(instruction, entry_date, test_prices, volatility, rates)
    budget = cfg.initial_capital * cfg.premium_budget_fraction
    sizing = budgeted_size(row.beta, spots, deltas, prices, budget,
        cfg.max_hedge_error, cfg.slippage_bps, cfg.commission_per_contract)
    values = dict(pair=row.pair, signal_date=row.signal_date, entry_date=entry_date, expiry_date=expiry_date,
        dependent_type=types[0], independent_type=types[1], dependent_price=prices[0], independent_price=prices[1],
        illustrative_budget=budget, status='priced' if sizing else 'no_feasible_integer_hedge')
    if sizing:
        values.update(sizing)
    rows.append(values)
option_preview = pd.DataFrame(rows)
save_frame('option_preview', option_preview)
display(option_preview)
if option_preview.empty:
    print('No snapshot candidates. Continue to Module 07 for the full daily experiment.')


## 4. Direction and expiry checks

Zero-dividend European options on adjusted synthetic spots remain an explicit model assumption. Costs are research assumptions, not measured option spreads.


In [ ]:
from src.backtest import black_scholes_price, option_types_from_spread_direction
assert option_types_from_spread_direction(1) == ('put', 'call')
assert option_types_from_spread_direction(-1) == ('call', 'put')
assert black_scholes_price(110, 100, 0, 0.03, 0.2, 'call') == 10
assert black_scholes_price(90, 100, 0, 0.03, 0.2, 'put') == 10
print('Direction and expiry payoff checks passed.')


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
print(f'Completed. Files saved in {OUTPUT_DIR}')
